# Fixed 12 starts versus stationarity-triggered starts

Corrected comparison on the same 50 coefficient-recovery processes from `ar3_minimum_width_k4.ipynb`.
Both modes use **max_it=2000**, **improv_patience=200**, improvement tolerance 1e-10, and the same ordered initial directions. Each attempt resets the original geometric step schedule.

- Fixed mode completes all 12 attempts and returns the smallest-width result, irrespective of stationarity.
- Conditional mode checks the best-width direction only after each run ends. It stops at the first stationary result, otherwise restarts from the next direction, capped at 12 attempts.
- There is **no stationary filtering of intermediate iterates**. The true coefficients do not enter decisions.

Stationarity uses the companion spectral radius < 1; see [statsmodels stationarity documentation](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima_process.ArmaProcess.isstationary.html) and [companion-root equivalence, Notes](https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.tools.is_invertible.html#notes).


In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'Code/Code_AR_processes/ar_tools/descent.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from output.analysis.ar3_conditional_restarts_2000_200.run_experiment import main as run_experiment
from output.analysis.ar3_conditional_restarts_2000_200.verify_and_report import validate, make_report
DATA_DIR = PROJECT_ROOT / 'Code/Code_AR_processes/outputs/csv/ar3_conditional_restarts_2000_200'


In [2]:
summary = run_experiment()


Completed 1/50 (0.0s)
Completed 5/50 (0.0s)
Completed 10/50 (0.0s)
Completed 15/50 (0.0s)
Completed 20/50 (0.0s)
Completed 25/50 (0.0s)
Completed 30/50 (0.0s)
Completed 35/50 (0.0s)
Completed 40/50 (0.0s)
Completed 45/50 (0.0s)
Completed 50/50 (0.0s)
                       method  processes  stationary_estimates  exhausted_without_stationary  mean_coefficient_mse  median_coefficient_mse  maximum_coefficient_mse  mse_above_1  mean_attempts  total_attempts  mean_evaluated_states  total_evaluated_states
              Fixed 12 starts         50                    46                             0            218.295726                0.000015              7680.880243            4          12.00             600               12000.10                  600005
Stationarity-triggered starts         50                    50                             0              0.030507                0.000038                 0.702086            0           1.66              83                1578.16         

In [3]:
validation = validate()
print(json.dumps(validation, indent=2))
figure = make_report()


{
  "processes": 50,
  "terminal_candidates_checked": 600,
  "policy_results_checked": 100,
  "selected_widths_checked": true,
  "coefficient_ratios_and_mses_checked": true,
  "independent_polynomial_root_check": true,
  "stopping_counts_checked": true,
  "first_stationary_selection_and_cap_checked": true,
  "complete_traces_checked": 48,
  "max_width_error": 0.0,
  "total_evaluated_states": {
    "Fixed 12 starts": 600005,
    "Stationarity-triggered starts": 78908
  },
  "filtering_used": false
}


The conditional rule reduces aggregate MSE by avoiding the four nonstationary fixed-mode outputs. It uses **83 attempts** instead of 600 and **78,908 iteration evaluations** instead of 600,005 (86.85% fewer). It also has higher coefficient MSE in 32 of the 50 pairs; its median MSE is 3.84e-5 versus 1.53e-5. Thus first-stationary acceptance trades accuracy in many individual processes for lower cost and removal of the severe nonstationary failures observed here.

![Paired accuracy and computation](outputs/images/ar3_conditional_restarts_2000_200/comparison.png)

[Full findings](outputs/csv/ar3_conditional_restarts_2000_200/findings.md) · [All 100 results](outputs/csv/ar3_conditional_restarts_2000_200/per_process_results.csv) · [Independent checks](outputs/csv/ar3_conditional_restarts_2000_200/validation.json) · [Export figure as PDF](outputs/pdfs/ar3_conditional_restarts_2000_200/comparison.pdf)
